In [16]:
import datetime as dt 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
import xarray as xr
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import RFE
from sklearn import metrics
from sklearn.linear_model import LinearRegression 

In [2]:
data_path = "F:\\Local Data\\ClimateBench\\"

# path to save the netcdf file
path_output = data_path + 'outputs_ssp245_prediction_pattern2.nc'

# path to testing data (ssp245)
pathssp245_Y = 'outputs_ssp245.nc'
pathssp245_X = 'inputs_ssp245.nc'

In [17]:
print(pathssp245_Y)

outputs_ssp245.nc


In [18]:
# from utils import load_inputs
ds = load_inputs("ssp585")
print(ds.dims)

FrozenMappingWarningOnValuesAccess({'time': 86, 'latitude': 96, 'longitude': 144})


In [19]:
# utils.py

import numpy as np
import xarray as xr
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH = Path("/home/vpaidi/private/Dsc-180/")

SECONDS_IN_YEAR = 60 * 60 * 24 * 365

# Ensemble member to use per scenario (historical member 1 has missing DTR)
MEMBER_SELECTION = {
    "historical": 2,
    "ssp126":     1,
    "ssp370":     1,
    "ssp585":     1,
    "ssp245":     None,   # test set — always average over all members
}

# ── Loaders ───────────────────────────────────────────────────────────────────
def load_inputs(scenario: str) -> xr.Dataset:
    """
    Load the input forcing dataset for a given scenario.
    Returns an xarray Dataset with variables CO2, CH4, SO2, BC.
    """
    path = DATA_PATH / f"inputs_{scenario}.nc"
    return xr.open_dataset(path).compute()


def load_targets(scenario: str, precip_to_mm: bool = True) -> xr.Dataset:
    """
    Load the output target dataset for a given scenario.

    - Selects the canonical ensemble member (see MEMBER_SELECTION).
    - For ssp245 (test), returns the ensemble mean across all members.
    - Converts pr and pr90 from kg/m2/s to mm/day if precip_to_mm=True.
    """
    path = DATA_PATH / f"outputs_{scenario}.nc"
    ds = xr.open_dataset(path).compute()

    member = MEMBER_SELECTION.get(scenario)
    if member is None:
        # test scenario — average over ensemble
        ds = ds.mean("member")
    else:
        ds = ds.sel(member=member)

    if precip_to_mm:
        ds["pr"]   = ds["pr"]   * 86400
        ds["pr90"] = ds["pr90"] * 86400

    return ds


# ── Weighting & spatial aggregation ──────────────────────────────────────────
def cos_weights(lat: np.ndarray) -> xr.DataArray:
    """
    Cosine-of-latitude area weights, normalised to sum to 1.
    Accepts a latitude array in degrees.
    """
    w = np.cos(np.deg2rad(lat))
    return w / w.sum()


def global_mean(field: xr.DataArray) -> xr.DataArray:
    """
    Area-weighted global mean over lat and lon.
    field must have dimensions that include 'lat' and 'lon'.
    Returns a DataArray with lat/lon averaged out.
    """
    w = np.cos(np.deg2rad(field.lat))
    return field.weighted(w).mean(["lat", "lon"])


# ── Training scenario list ────────────────────────────────────────────────────
TRAIN_SCENARIOS = ["historical", "ssp585", "ssp126", "ssp370"]
TEST_SCENARIO   = "ssp245"

In [20]:
# pattern_scaling.py

import numpy as np
import xarray as xr
import pandas as pd
from sklearn.linear_model import LinearRegression


TARGETS = ["tas", "pr", "pr90", "diurnal_temperature_range"]

class PatternScalingModel:

    def __init__(self):
        self.regressors = {}       # one LinearRegression per target
        self.gmt_regressor = None  # for input-driven GMT prediction
        self._lat = None           # stored for output coords
        self._lon = None
        self._training_gmt = None  # stored for diagnostics

    # ── Fit ───────────────────────────────────────────────────────────────────
    def fit(self, scenarios=TRAIN_SCENARIOS):
        """
        Fit pattern scaling on the given training scenarios.
        Concatenates all scenarios along time before fitting.
        """
        print("Loading training data...")
        Y_list, X_list = [], []
        for s in scenarios:
            Y_list.append(load_targets(s))
            X_list.append(load_inputs(s))

        Y = xr.concat(Y_list, dim="time")
        X = xr.concat(X_list, dim="time")

        # Reset time to integer index to avoid datetime conflicts
        n = len(Y.time)
        Y["time"] = np.arange(n)
        X["time"] = np.arange(n)

        # Store coords for later output reconstruction
        self._lat = Y.lat.values
        self._lon = Y.lon.values

        # Training GMT — the sole predictor for pattern scaling
        gmt = global_mean(Y["tas"]).values  # (n_time,)
        self._training_gmt = gmt

        print(f"Fitting on {n} timesteps across {len(scenarios)} scenarios...")

        # Fit one LinearRegression per target
        for target in TARGETS:
            y_stacked = Y[target].values.reshape(n, -1)  # (n_time, n_cells)
            reg = LinearRegression(fit_intercept=False)
            reg.fit(gmt.reshape(-1, 1), y_stacked)
            self.regressors[target] = reg

In [21]:
def nrmse(truth_full, pred_full):
    """
    NRMSE per ClimateBench paper:
    - RMSE computed spatially (area-weighted) then averaged over 2080-2100
    - Normalised by std of the full spatiotemporal truth field over 2080-2100
    """
    w = np.cos(np.deg2rad(truth_full.lat))
    w_norm = w / w.sum()  # normalised weights, sum to 1

    truth_eval = truth_full.sel(time=slice(2080, 2101))
    pred_eval  = pred_full.sel(time=slice(2080, 2101))

    # Spatial RMSE per year, then mean over years
    diff = truth_eval - pred_eval
    spatial_rmse = np.sqrt(
        ((diff**2) * w_norm).sum(["lat", "lon"])
    ).mean("time")

    # Std of truth over full spatiotemporal field (weighted)
    norm = float(np.sqrt(
        ((truth_eval - truth_eval.weighted(w).mean(["lat","lon","time"]))**2 
         * w_norm).sum(["lat", "lon"]).mean("time")
    ))

    return float(spatial_rmse / norm)

print(f"{'Target':<30} {'Oracle NRMSE':>14} {'Input-driven NRMSE':>20}")
print("-" * 66)
for target in TARGETS:
    s_oracle = nrmse(test_Y[target], pred_oracle[target])
    s_inputs = nrmse(test_Y[target], pred_inputs[target])
    print(f"{target:<30} {s_oracle:>14.4f} {s_inputs:>20.4f}")

Target                           Oracle NRMSE   Input-driven NRMSE
------------------------------------------------------------------
tas                                    0.3047               0.3058
pr                                     0.7703               0.7702
pr90                                   0.7800               0.7799
diurnal_temperature_range              0.4878               0.4877


In [23]:


import numpy as np
import xarray as xr
import pandas as pd
from sklearn.linear_model import LinearRegression

TARGETS = ["tas", "pr", "pr90", "diurnal_temperature_range"]

# ── Load training data ────────────────────────────────────────────────────────
print("Loading training data...")
Y = xr.concat([load_targets(s) for s in TRAIN_SCENARIOS], dim="time")
X = xr.concat([load_inputs(s) for s in TRAIN_SCENARIOS], dim="time")

n = len(Y.time)
Y["time"] = np.arange(n)
X["time"] = np.arange(n)

gmt_train = global_mean(Y["tas"]).values  # (n_time,)
print(f"Training timesteps: {n}")

# ── Load test data ────────────────────────────────────────────────────────────
print("Loading test data...")
test_Y = load_targets("ssp245")
test_X = load_inputs("ssp245")

# Oracle GMT — from test outputs (cheating)
gmt_oracle = global_mean(test_Y["tas"]) \
               .to_series() \
               .rolling(10, min_periods=1).mean() \
               .values

# Input-driven GMT — from CO2 only (honest)
gmt_reg = LinearRegression()
gmt_reg.fit(X["CO2"].values.reshape(-1, 1), gmt_train)
gmt_predicted = pd.Series(
    gmt_reg.predict(test_X["CO2"].values.reshape(-1, 1)).flatten()
).rolling(10, min_periods=1).mean().values

# ── Fit one regressor per target ──────────────────────────────────────────────
print("Fitting pattern scaling...")
regressors = {}
for target in TARGETS:
    y_stacked = Y[target].values.reshape(n, -1)
    reg = LinearRegression(fit_intercept=False)
    reg.fit(gmt_train.reshape(-1, 1), y_stacked)
    regressors[target] = reg
    print(f"  Fitted {target}")

# ── Build output datasets ─────────────────────────────────────────────────────
def make_output(gmt, time_coords):
    n_test = len(gmt)
    lat, lon = test_Y.lat.values, test_Y.lon.values
    ds = xr.Dataset(coords={"time": time_coords, "lat": lat, "lon": lon})
    for target, reg in regressors.items():
        pred = reg.predict(gmt.reshape(-1, 1))
        ds[target] = (["time", "lat", "lon"],
                      pred.reshape(n_test, len(lat), len(lon)))
    return ds

pred_oracle = make_output(gmt_oracle, test_X.time.values)
pred_inputs = make_output(gmt_predicted, test_X.time.values)

# ── NRMSE ─────────────────────────────────────────────────────────────────────
def nrmse(truth_full, pred_full):
    w = np.cos(np.deg2rad(truth_full.lat))
    w_norm = w / w.sum()
    truth_eval = truth_full.sel(time=slice(2080, 2101))
    pred_eval  = pred_full.sel(time=slice(2080, 2101))
    diff = truth_eval - pred_eval
    spatial_rmse = np.sqrt(
        ((diff**2) * w_norm).sum(["lat", "lon"])
    ).mean("time")
    norm = float(np.sqrt(
        ((truth_eval - truth_eval.weighted(w).mean(["lat", "lon", "time"]))**2
         * w_norm).sum(["lat", "lon"]).mean("time")
    ))
    return float(spatial_rmse / norm)

# ── Scores ────────────────────────────────────────────────────────────────────
print(f"\n{'Target':<30} {'Oracle NRMSE':>14} {'Input-driven NRMSE':>20}")
print("-" * 66)
for target in TARGETS:
    s_oracle = nrmse(test_Y[target], pred_oracle[target])
    s_inputs = nrmse(test_Y[target], pred_inputs[target])
    print(f"{target:<30} {s_oracle:>14.4f} {s_inputs:>20.4f}")



Loading training data...


/tmp/ipykernel_1858/1902033696.py:10: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  Y = xr.concat([load_targets(s) for s in TRAIN_SCENARIOS], dim="time")


Training timesteps: 423
Loading test data...
Fitting pattern scaling...
  Fitted tas
  Fitted pr
  Fitted pr90
  Fitted diurnal_temperature_range

Target                           Oracle NRMSE   Input-driven NRMSE
------------------------------------------------------------------
tas                                    0.3047               0.3058
pr                                     0.7703               0.7702
pr90                                   0.7800               0.7799
diurnal_temperature_range              0.4878               0.4877


In [24]:
import jax
import jax.numpy as jnp
print(jax.__version__)
print(jnp.array([1.0, 2.0, 3.0]))

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


0.10.2
[1. 2. 3.]


In [25]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)  # match float64 precision of sklearn

# ── JAX fit ───────────────────────────────────────────────────────────────────
gmt_train_jax = jnp.array(gmt_train).reshape(-1, 1)

jax_patterns = {}
for target in TARGETS:
    y_stacked = jnp.array(Y[target].values.reshape(n, -1))
    alpha, _, _, _ = jnp.linalg.lstsq(gmt_train_jax, y_stacked, rcond=None)
    jax_patterns[target] = alpha
    print(f"Fitted {target} (JAX)")

# ── JIT-compiled predict ──────────────────────────────────────────────────────
@jax.jit
def jax_predict(gmt, alpha):
    """gmt: (n,1)  alpha: (1, n_cells)  →  (n, n_cells)"""
    return gmt @ alpha

# ── Oracle prediction ─────────────────────────────────────────────────────────
gmt_oracle_jax = jnp.array(gmt_oracle).reshape(-1, 1)

pred_jax = xr.Dataset(coords={"time": test_X.time.values,
                                "lat":  test_Y.lat.values,
                                "lon":  test_Y.lon.values})
for target in TARGETS:
    pred = np.array(jax_predict(gmt_oracle_jax, jax_patterns[target]))
    pred_jax[target] = (["time", "lat", "lon"],
                         pred.reshape(86, len(test_Y.lat), len(test_Y.lon)))

# ── Compare scores ────────────────────────────────────────────────────────────
print(f"\n{'Target':<30} {'sklearn':>10} {'JAX':>10} {'Delta':>10}")
print("-" * 62)
for target in TARGETS:
    s_sk  = nrmse(test_Y[target], pred_oracle[target])
    s_jax = nrmse(test_Y[target], pred_jax[target])
    print(f"{target:<30} {s_sk:>10.4f} {s_jax:>10.4f} {abs(s_sk-s_jax):>10.6f}")

Fitted tas (JAX)
Fitted pr (JAX)
Fitted pr90 (JAX)
Fitted diurnal_temperature_range (JAX)

Target                            sklearn        JAX      Delta
--------------------------------------------------------------
tas                                0.3047     0.3047   0.000000
pr                                 0.7703     0.7703   0.000000
pr90                               0.7800     0.7800   0.000000
diurnal_temperature_range          0.4878     0.4878   0.000000
